In [ ]:
"""
Interactive Polygon Drawing Tool for Land Cover Classification
Интерактивен инструмент за чертане на полигони за класификация на земното покритие

This script:
- Finds all TIFF images starting with 'B' (high-resolution data) in the specified input directory.
- For each image, lets the user draw training polygons for 5 merged land cover classes.
- Polygons are saved as GeoPackage files, both per image/class and in a combined file.

Този скрипт:
- Намира всички TIFF изображения, започващи с 'B' (високодетайлни данни) в посочената входна директория.
- За всяко изображение позволява на потребителя да чертае обучителни полигони за 5 обединени класа земно покритие.
- Полигоните се записват като GeoPackage файлове, както по изображение/клас, така и в общ файл.

Updated paths (обновени пътища):
  high_res_dir = D:\data\master_thesis\input\high_resolution_data
  output_dir = D:\data\master_thesis\exports\object_classification_supervised
  POLYGON_BASE_DIR = D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures\polygons
  REFERENCE_TIF_PATH = D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures\fire4_simple_average_all_bands_2024.tif
"""

import rasterio
import matplotlib
# Force matplotlib to use an interactive backend that opens new windows
# Принудително задава интерактивен бекенд на matplotlib, който отваря нови прозорци
matplotlib.use('Qt5Agg')  # This will open new windows for drawing
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Import for polygon handling / Импортиране за работа с полигони
import geopandas as gpd
from shapely.geometry import Polygon, Point
import pandas as pd

# Set directory paths / Задаване на пътищата към директориите
high_res_dir = r"D:\data\master_thesis\input\high_resolution_data"
output_dir = r"D:\data\master_thesis\exports\object_classification_supervised"
POLYGON_BASE_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures\polygons'
REFERENCE_TIF_PATH = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures\fire4_simple_average_all_bands_2024.tif'

os.makedirs(output_dir, exist_ok=True)

# ==================== FIXED CLASS DEFINITIONS ====================
# Here we define the merged land cover classes with their colors.
# Тук дефинираме обединените класове земно покритие с техните цветове.
CLASSES = {
    1: {'name': 'Urban / Built-up / Bare Territories', 'color': (255, 0, 0), 'hex': '#FF0000'},
    2: {'name': 'Water', 'color': (0, 0, 255), 'hex': '#0000FF'},
    3: {'name': 'Field/Agriculture', 'color': (255, 255, 0), 'hex': '#FFFF00'},
    4: {'name': 'Coniferous Forest', 'color': (0, 100, 0), 'hex': '#006400'},
    5: {'name': 'Deciduous Forest', 'color': (0, 255, 0), 'hex': '#00FF00'}
}

# Original class mapping for reference (before merging)
# Оригинално съпоставяне на класовете за справка (преди обединяване)
ORIGINAL_CLASSES = {
    1: {'name': 'Urban', 'color': (255, 0, 0), 'polygon_file': 'urban'},
    2: {'name': 'Water', 'color': (0, 0, 255), 'polygon_file': 'water'},
    3: {'name': 'Bare and Urban Territories', 'color': (139, 69, 19), 'polygon_file': 'bare_lands'},
    4: {'name': 'Field/Agriculture', 'color': (255, 255, 0), 'polygon_file': 'field_agriculture'},
    5: {'name': 'Coniferous Forest', 'color': (0, 100, 0), 'polygon_file': 'coniferous'},
    6: {'name': 'Deciduous Forest', 'color': (0, 255, 0), 'polygon_file': 'forest_deciduous'}
}

# Mapping from original class IDs to merged class IDs
# Съпоставяне от оригиналните идентификатори на класове към обединените
CLASS_MAPPING = {
    1: 1,  # Urban -> Urban/Bare
    2: 2,  # Water -> Water
    3: 1,  # Bare/Urban -> Urban/Bare (MERGED)
    4: 3,  # Field -> Field
    5: 4,  # Coniferous -> Coniferous
    6: 5   # Deciduous -> Deciduous
}

# ==================== SIMPLIFIED INTERACTIVE POLYGON DRAWING TOOL ====================
# Опростен интерактивен инструмент за чертане на полигони

class SimplePolygonDrawer:
    """
    Simplified interactive tool for drawing training polygons.
    Uses downsampling for large images.

    Опростен интерактивен инструмент за чертане на обучителни полигони.
    Използва намаляване на разделителната способност за големи изображения.
    """
    
    def __init__(self, image_rgb, src, class_id, class_name, image_name, output_dir, display_scale=4):
        """
        Initialize the simple polygon drawer.
        Инициализиране на инструмента за чертане.

        Parameters / Параметри:
        -----------
        display_scale : int
            Downsampling factor for display (to handle large images)
            Фактор на намаляване за показване (за работа с големи изображения)
        """
        self.image_rgb = image_rgb
        self.src = src
        self.class_id = class_id
        self.class_name = class_name
        self.image_name = image_name
        self.output_dir = output_dir
        self.display_scale = display_scale
        
        self.polygons = []  # Store drawn polygons in geographic coordinates
                            # Съхраняване на начертаните полигони в географски координати
        self.current_polygon_pts = []  # Current polygon being drawn in display pixel coordinates
                                       # Текущ полигон, който се чертае в пикселни координати на екрана
        self.fig = None
        self.ax = None
        self.cid_click = None
        self.cid_key = None
        self.quit_drawing = False
        
    def display_to_geo(self, x_display, y_display):
        """Convert display coordinates to geographic coordinates.
           Преобразуване от екранни координати в географски."""
        # Convert display coordinates to original pixel coordinates
        # Конвертиране на екранни координати в оригинални пикселни координати
        x_orig = x_display * self.display_scale
        y_orig = y_display * self.display_scale
        
        # Convert to geographic coordinates / Преобразуване в географски координати
        x_geo, y_geo = rasterio.transform.xy(self.src.transform, y_orig, x_orig)
        return x_geo, y_geo
    
    def draw_polygons(self):
        """
        Launch interactive polygon drawing interface.
        Returns True if polygons were drawn, False otherwise.

        Стартира интерактивния интерфейс за чертане на полигони.
        Връща True, ако са начертани полигони, в противен случай False.
        """
        print(f"\n  🎨 Drawing polygons for class: {self.class_name}")
        print(f"  Image: {self.image_name}")
        print(f"  Display scale: 1:{self.display_scale} (downsampled for viewing)")
        print("  Instructions / Инструкции:")
        print("    - LEFT CLICK to add points / ЛЯВ БУТОН за добавяне на точки")
        print("    - RIGHT CLICK to close polygon / ДЕСЕН БУТОН за затваряне на полигона")
        print("    - Press 's' to SKIP this class / Натиснете 's' за ПРОПУСКАНЕ на този клас")
        print("    - Press 'q' to QUIT drawing for this image / Натиснете 'q' за ИЗХОД от чертането за това изображение")
        print("    - CLOSE WINDOW to skip/move to next class / ЗАТВОРЕТЕ ПРОЗОРЕЦА за пропускане/преминаване към следващ клас")
        
        # Create a new figure - this will open a new window
        # Създаване на нова фигура – ще отвори нов прозорец
        self.fig, self.ax = plt.subplots(figsize=(12, 10))
        self.ax.imshow(self.image_rgb)
        self.ax.set_title(f"CLASS: {self.class_name.upper()}\nL-Click: Points | R-Click: Finish Poly | 's': Skip Class | 'q': Quit Image", 
                         fontsize=12)
        self.ax.axis('off')
        
        # Connect click event / Свързване на събитие при кликване
        self.cid_click = self.fig.canvas.mpl_connect('button_press_event', self.on_click)
        
        # Connect key press event / Свързване на събитие при натискане на клавиш
        self.cid_key = self.fig.canvas.mpl_connect('key_press_event', self.on_key)
        
        plt.tight_layout()
        
        # This will block execution until the window is closed
        # Това ще блокира изпълнението, докато прозорецът не бъде затворен
        plt.show(block=True)
        
        # After closing, save polygons if any were drawn
        # След затваряне записва полигоните, ако има такива
        if self.polygons:
            self.save_polygons()
            return True
        else:
            print(f"    No polygons drawn for {self.class_name}")
            return False
    
    def on_click(self, event):
        """Handle mouse clicks. / Обработка на кликванията с мишката."""
        if event.inaxes != self.ax:
            return
        
        if event.button == 1:  # Left click - add point / Ляв бутон – добавяне на точка
            self.current_polygon_pts.append((int(event.xdata), int(event.ydata)))
            self.update_display()
            
        elif event.button == 3:  # Right click - close polygon / Десен бутон – затваряне на полигона
            if len(self.current_polygon_pts) >= 3:
                # Close the polygon / Затваряне на полигона
                if self.current_polygon_pts[0] != self.current_polygon_pts[-1]:
                    self.current_polygon_pts.append(self.current_polygon_pts[0])
                
                # Draw the completed polygon on display / Начертаване на завършения полигон на екрана
                x, y = zip(*self.current_polygon_pts)
                self.ax.plot(x, y, color='cyan', linewidth=2)
                self.ax.fill(x, y, alpha=0.3, color='yellow')
                
                # Convert to geographic coordinates / Преобразуване в географски координати
                geo_coords = []
                for px_disp, py_disp in self.current_polygon_pts:
                    x_geo, y_geo = self.display_to_geo(px_disp, py_disp)
                    geo_coords.append((x_geo, y_geo))
                
                # Create polygon / Създаване на полигон
                poly = Polygon(geo_coords)
                if poly.is_valid and not poly.is_empty:
                    self.polygons.append(poly)
                    print(f"      Polygon added. Total: {len(self.polygons)}")
                
                # Clear current polygon / Изчистване на текущия полигон
                self.current_polygon_pts = []
                
                self.fig.canvas.draw()
            else:
                print("      Need at least 3 points to close polygon")
    
    def on_key(self, event):
        """Handle key presses. / Обработка на натискания на клавиши."""
        if event.key == 's':  # 's' key - skip this class / клавиш 's' – пропускане на този клас
            print(f"    Skipping class {self.class_name}")
            plt.close(self.fig)
        elif event.key == 'q':  # 'q' key - quit drawing for this image / клавиш 'q' – изход от чертането за това изображение
            print(f"    Quitting drawing for this image")
            self.quit_drawing = True
            plt.close(self.fig)
    
    def update_display(self):
        """Update the display with current polygon.
           Обновяване на екрана с текущия полигон."""
        # Clear previous drawings (except the base image)
        # Изчистване на предишните чертежи (без основното изображение)
        for artist in self.ax.lines + self.ax.collections:
            artist.remove()
        
        # Redraw all completed polygons / Пречертаване на всички завършени полигони
        for poly in self.polygons:
            # Convert back to display coordinates for display
            # Обратно преобразуване в екранни координати за показване
            display_coords = []
            for x_geo, y_geo in poly.exterior.coords:
                # Convert geographic to original pixel coordinates
                # Преобразуване от географски в оригинални пикселни координати
                row_orig, col_orig = rasterio.transform.rowcol(self.src.transform, x_geo, y_geo)
                # Convert to display coordinates / Преобразуване в екранни координати
                px_disp = col_orig // self.display_scale
                py_disp = row_orig // self.display_scale
                display_coords.append((px_disp, py_disp))
            
            if len(display_coords) >= 3:
                x, y = zip(*display_coords)
                self.ax.plot(x, y, color='cyan', linewidth=2)
                self.ax.fill(x, y, alpha=0.3, color='yellow')
        
        # Draw current polygon / Чертане на текущия полигон
        if len(self.current_polygon_pts) >= 2:
            x, y = zip(*self.current_polygon_pts)
            self.ax.plot(x, y, color='yellow', marker='o', markersize=3, linewidth=1)
        elif len(self.current_polygon_pts) == 1:
            x, y = self.current_polygon_pts[0]
            self.ax.plot(x, y, 'yo', markersize=3)
        
        self.fig.canvas.draw()
    
    def save_polygons(self):
        """Save drawn polygons to file.
           Записване на начертаните полигони във файл."""
        if not self.polygons:
            return
        
        # Create data dictionary with lists / Създаване на речник с данни със списъци
        data = {
            'class_id': [self.class_id] * len(self.polygons),
            'class_name': [self.class_name] * len(self.polygons),
            'image_name': [self.image_name] * len(self.polygons)
        }
        
        # Create GeoDataFrame with explicit index / Създаване на GeoDataFrame с явен индекс
        gdf = gpd.GeoDataFrame(
            data,
            geometry=self.polygons,
            crs=self.src.crs,
            index=range(len(self.polygons))
        )
        
        # Save to file in the main output directory / Запис във файл в основната изходна директория
        safe_class_name = self.class_name.lower().replace(' ', '_').replace('/', '_').replace('-', '_')
        safe_image_name = os.path.splitext(self.image_name)[0].replace(' ', '_')
        filename = f"{safe_image_name}_{safe_class_name}_drawn.gpkg"
        filepath = os.path.join(self.output_dir, filename)
        gdf.to_file(filepath, driver='GPKG')
        print(f"    ✅ Saved {len(self.polygons)} polygons to {filename}")
        
        # Also append to a combined file for all images
        # Също така добавяне към общ файл за всички изображения
        combined_filename = f"all_training_polygons_combined.gpkg"
        combined_path = os.path.join(self.output_dir, combined_filename)
        
        if os.path.exists(combined_path):
            # Append to existing file / Добавяне към съществуващ файл
            existing_gdf = gpd.read_file(combined_path)
            combined_gdf = pd.concat([existing_gdf, gdf], ignore_index=True)
            combined_gdf.to_file(combined_path, driver='GPKG')
        else:
            # Create new file / Създаване на нов файл
            gdf.to_file(combined_path, driver='GPKG')
        
        print(f"    ✅ Also appended to combined file: {combined_filename}")

# ==================== HELPER FUNCTIONS ====================
# Помощни функции

def load_reference_crs():
    """Load CRS from the reference GeoTIFF file.
       Зареждане на координатна система от референтния GeoTIFF файл."""
    print(f"\n📌 Loading reference CRS from: {REFERENCE_TIF_PATH}")
    
    if not os.path.exists(REFERENCE_TIF_PATH):
        print(f"❌ Reference file not found: {REFERENCE_TIF_PATH}")
        return None, None, None
    
    try:
        with rasterio.open(REFERENCE_TIF_PATH) as src:
            crs = src.crs
            transform = src.transform
            bounds = src.bounds
            print(f"✅ Loaded reference CRS: {crs}")
            print(f"✅ Reference transform: {transform}")
            print(f"✅ Reference bounds: {bounds}")
            return crs, transform, bounds
    except Exception as e:
        print(f"❌ Error loading reference file: {e}")
        return None, None, None

def find_d_tif_files(directory):
    """
    Find all .tif and .tiff files starting with 'B' in the directory.
    Exclude .aux.xml and other non-image files.

    Намира всички .tif и .tiff файлове, започващи с 'B' в директорията.
    Изключва .aux.xml и други не-изображения.
    """
    print(f"\n🔍 Searching for TIFF files starting with 'B' in: {directory}")
    
    if not os.path.exists(directory):
        print(f"❌ Directory does not exist: {directory}")
        return []
    
    # Method 1: Using os.listdir (simpler and more reliable)
    # Метод 1: Използване на os.listdir (по-прост и надежден)
    tif_files = []
    
    # Get all files in directory / Вземане на всички файлове в директорията
    all_files = os.listdir(directory)
    
    for filename in all_files:
        # Check if file starts with 'D' or 'd' and has a TIFF extension
        # Проверка дали файлът започва с 'B' (главна или малка буква) и има разширение TIFF
        if filename.upper().startswith('B') and filename.lower().endswith(('.tif', '.tiff')):
            full_path = os.path.join(directory, filename)
            # Exclude .aux.xml files / Изключване на .aux.xml файлове
            if not filename.endswith('.aux.xml') and not '.aux.xml' in filename:
                tif_files.append(full_path)
    
    tif_files.sort()
    
    if tif_files:
        print(f"  ✅ Total D* TIFF files found: {len(tif_files)}")
        print(f"  Files found:")
        for f in tif_files[:10]:
            print(f"    - {os.path.basename(f)}")
        if len(tif_files) > 10:
            print(f"    ... and {len(tif_files) - 10} more")
    else:
        print(f"  ❌ No TIFF files starting with 'D' found")
        # Show all TIFF files in directory for debugging
        # Показване на всички TIFF файлове в директорията за дебъгване
        print(f"\n  Other TIFF files in directory:")
        tif_in_dir = [f for f in all_files if f.lower().endswith(('.tif', '.tiff'))]
        if tif_in_dir:
            for f in tif_in_dir[:15]:
                print(f"    - {f}")
    
    return tif_files

def check_image_crs(image_path):
    """Check the CRS of a specific image.
       Проверка на координатната система на конкретно изображение."""
    try:
        with rasterio.open(image_path) as src:
            print(f"  Image CRS: {src.crs}")
            print(f"  Image bounds: {src.bounds}")
            return src.crs
    except Exception as e:
        print(f"  Error reading CRS: {e}")
        return None

def create_downsampled_rgb_display(file_path, max_display_size=2000):
    """
    Create a downsampled RGB display image to avoid memory issues.
    Създава намалено RGB изображение за показване, за да се избегнат проблеми с паметта.

    Parameters / Параметри:
    -----------
    file_path : str
        Path to the image file / Път до файла с изображението
    max_display_size : int
        Maximum size for display (width or height) / Максимален размер за показване (ширина или височина)

    Returns / Връща:
    --------
    tuple : (downsampled_rgb, scale_factor, src)
    """
    with rasterio.open(file_path) as src:
        # Calculate scale factor to limit display size
        # Изчисляване на мащабен фактор за ограничаване на размера на дисплея
        max_dim = max(src.width, src.height)
        scale_factor = max(1, max_dim // max_display_size)
        
        print(f"    Creating downsampled display (scale 1:{scale_factor})")
        
        # Read RGB bands with downsampling / Четене на RGB канали с намаляване
        out_shape = (src.count, src.height // scale_factor, src.width // scale_factor)
        
        # Read all bands / Четене на всички канали
        img_data = src.read(out_shape=out_shape)
        
        # If we have more than 3 bands, take first 3 for RGB
        # Ако има повече от 3 канала, вземаме първите 3 за RGB
        if img_data.shape[0] > 3:
            # Try to use typical RGB band order (4,3,2 for Sentinel-2)
            # Опит за използване на типичния ред за RGB (4,3,2 за Sentinel-2)
            try:
                img_data = src.read([4, 3, 2], out_shape=(3, src.height // scale_factor, src.width // scale_factor))
            except:
                # Fallback to first 3 bands / Резервен вариант – първите 3 канала
                img_data = src.read([1, 2, 3], out_shape=(3, src.height // scale_factor, src.width // scale_factor))
        elif img_data.shape[0] == 1:
            # Single band - duplicate to create RGB
            # Единичен канал – дублиране за създаване на RGB
            img_data = np.repeat(img_data, 3, axis=0)
        elif img_data.shape[0] == 2:
            # Two bands - pad with zeros / Два канала – допълване с нули
            img_data = np.vstack([img_data, np.zeros_like(img_data[0:1])])
        
        # Create display image with percentile stretch
        # Създаване на изображение за показване с персентилно разтягане
        img_rgb = np.transpose(img_data[:3], (1, 2, 0)).astype(float)
        
        # Apply percentile stretch for better visualization
        # Прилагане на персентилно разтягане за по-добра визуализация
        for i in range(3):
            p2, p98 = np.percentile(img_rgb[:,:,i][img_rgb[:,:,i] > 0], (2, 98)) if np.any(img_rgb[:,:,i] > 0) else (0, 1)
            if p98 > p2:
                img_rgb[:,:,i] = np.clip((img_rgb[:,:,i] - p2) / (p98 - p2), 0, 1)
        
        return img_rgb, scale_factor, src

def view_existing_polygons(output_dir):
    """
    View summary of existing training polygons.
    Преглед на обобщение на съществуващите обучителни полигони.
    """
    combined_filename = "all_training_polygons_combined.gpkg"
    combined_path = os.path.join(output_dir, combined_filename)
    
    if not os.path.exists(combined_path):
        print(f"\n📭 No existing training polygons found.")
        return None
    
    try:
        print(f"\n📂 Loading existing training polygons from: {combined_filename}")
        gdf = gpd.read_file(combined_path)
        
        if len(gdf) == 0:
            print("  ⚠ Combined file is empty")
            return None
        
        print(f"\n✅ Found {len(gdf)} total polygons")
        print(f"\n📊 Summary by class / Обобщение по клас:")
        
        # Group by class / Групиране по клас
        class_summary = gdf.groupby(['class_id', 'class_name']).size().reset_index(name='count')
        for _, row in class_summary.iterrows():
            print(f"  {row['class_name']}: {row['count']} polygons")
        
        print(f"\n📊 Summary by image / Обобщение по изображение:")
        image_summary = gdf.groupby('image_name').size().reset_index(name='count')
        for _, row in image_summary.iterrows():
            print(f"  {row['image_name']}: {row['count']} polygons")
        
        return gdf
        
    except Exception as e:
        print(f"  ❌ Error loading existing polygons: {e}")
        return None

# ==================== INTERACTIVE POLYGON DRAWING FOR D* FILES ====================
# Интерактивно чертане на полигони за файлове, започващи с D

def draw_polygons_for_d_files(tif_files, output_dir):
    """
    Interactive polygon drawing for all D* TIFF files.
    Allows drawing polygons for each class, with ability to skip classes.

    Интерактивно чертане на полигони за всички TIFF файлове, започващи с D.
    Позволява чертане на полигони за всеки клас, с възможност за пропускане на класове.

    Returns / Връща:
    --------
    bool: True if polygons were drawn, False otherwise
          True, ако са начертани полигони, False в противен случай
    """
    print(f"\n{'='*70}")
    print("🎨 INTERACTIVE POLYGON DRAWING FOR D* TIFF FILES")
    print(f"{'='*70}")
    print(f"Found {len(tif_files)} D* TIFF files to process")
    print(f"Polygons will be saved to: {output_dir}")
    print("\nInstructions for each class / Инструкции за всеки клас:")
    print("  - LEFT CLICK to add polygon vertices / ЛЯВ БУТОН за добавяне на върхове")
    print("  - RIGHT CLICK to close the polygon / ДЕСЕН БУТОН за затваряне на полигона")
    print("  - Press 's' to SKIP the current class / Натиснете 's' за ПРОПУСКАНЕ на текущия клас")
    print("  - Press 'q' to QUIT drawing for the current image / Натиснете 'q' за ИЗХОД от чертането за текущото изображение")
    print("  - Close window to move to next class / Затворете прозореца за преминаване към следващ клас")
    
    total_polygons_drawn = 0
    
    # Process each D* TIFF file / Обработка на всеки TIFF файл, започващ с D
    for file_idx, file_path in enumerate(tif_files):
        file_name = os.path.basename(file_path)
        
        print(f"\n{'='*70}")
        print(f"📸 [{file_idx+1}/{len(tif_files)}] Processing image: {file_name}")
        print(f"{'='*70}")
        
        # Check image CRS / Проверка на координатната система
        print(f"\n🔍 Checking image CRS:")
        img_crs = check_image_crs(file_path)
        
        try:
            # Create downsampled display image / Създаване на намалено изображение за показване
            print(f"\n🖼️  Creating display image...")
            img_rgb, scale_factor, src = create_downsampled_rgb_display(file_path, max_display_size=2000)
            
            # Draw polygons for each class / Чертане на полигони за всеки клас
            quit_drawing = False
            image_polygons = 0
            
            for class_id, class_info in CLASSES.items():
                if quit_drawing:
                    break
                    
                class_name = class_info['name']
                
                print(f"\n  {'-'*50}")
                print(f"  CLASS {class_id}: {class_name}")
                print(f"  {'-'*50}")
                
                # Create drawer / Създаване на инструмента за чертане
                drawer = SimplePolygonDrawer(
                    img_rgb, src, class_id, class_name,
                    file_name, output_dir,
                    display_scale=scale_factor
                )
                
                # Draw polygons / Чертане на полигони
                if drawer.draw_polygons():
                    image_polygons += len(drawer.polygons)
                    total_polygons_drawn += len(drawer.polygons)
                    
                if drawer.quit_drawing:
                    print(f"  Quitting drawing for this image")
                    quit_drawing = True
                    break
            
            print(f"\n  ✅ Finished processing {file_name} - Drew {image_polygons} polygons")
            
        except Exception as e:
            print(f"  ❌ Error processing {file_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"\n{'='*70}")
    print(f"🎯 DRAWING COMPLETE / ЧЕРТАНЕТО ЗАВЪРШИ")
    print(f"{'='*70}")
    print(f"✅ Total polygons drawn: {total_polygons_drawn}")
    print(f"📁 All polygon files saved in: {output_dir}")
    
    # Show summary of all polygons / Показване на обобщение на всички полигони
    view_existing_polygons(output_dir)
    
    return total_polygons_drawn > 0

# ==================== MAIN EXECUTION ====================
# Основно изпълнение

def main():
    """Main execution function - only drawing polygons.
       Основна функция за изпълнение – само чертане на полигони."""
    print("="*70)
    print("INTERACTIVE POLYGON DRAWING FOR D* TIFF FILES")
    print("="*70)
    print(f"Classes to draw / Класове за чертане:")
    for class_id, class_info in CLASSES.items():
        print(f"  {class_id}: {class_info['name']}")
    print(f"\nInput directory / Входна директория: {high_res_dir}")
    print(f"Output directory / Изходна директория: {output_dir}")
    print("="*70)
    
    # Create output directory / Създаване на изходна директория
    os.makedirs(output_dir, exist_ok=True)
    
    # Step 1: Load reference CRS (optional, just for info)
    # Стъпка 1: Зареждане на референтна координатна система (по избор, само за информация)
    print("\n📌 Loading reference CRS (for information only)...")
    reference_crs, _, _ = load_reference_crs()
    
    # Step 2: Find D* TIFF files / Стъпка 2: Намиране на TIFF файлове, започващи с D
    print("\n📌 Finding D* TIFF files...")
    tif_files = find_d_tif_files(high_res_dir)
    
    if not tif_files:
        print("❌ No D* TIFF files found. Exiting.")
        return
    
    # Step 3: Check for existing polygons / Стъпка 3: Проверка за съществуващи полигони
    print("\n📌 Checking for existing polygons...")
    existing_gdf = view_existing_polygons(output_dir)
    
    if existing_gdf is not None:
        print("\n📝 Options / Опции:")
        print("  1. Draw additional polygons (will append to existing) / Чертане на допълнителни полигони (ще се добавят към съществуващите)")
        print("  2. Exit / Изход")
        choice = input("\nEnter choice (1 or 2): ").strip()
        
        if choice == '2':
            print("Exiting...")
            return
    
    # Step 4: Draw polygons / Стъпка 4: Чертане на полигони
    print("\n📌 Starting interactive polygon drawing...")
    polygons_drawn = draw_polygons_for_d_files(tif_files, output_dir)
    
    if polygons_drawn:
        print(f"\n✅ Successfully drew polygons!")
        print(f"📁 All polygons saved in: {output_dir}")
        
        # Show final summary / Показване на крайно обобщение
        view_existing_polygons(output_dir)
    else:
        print("\n❌ No polygons were drawn.")
    
    print("\n🎯 Drawing complete! You can now use these polygons for training.")
    print("="*70)


if __name__ == "__main__":
    main()